# Lab 17 · Optimisation 2: teach the model, on a small memory budget

**Time:** 60 min · **Work in teams of 3**

**Where we are in the day:** Lab 16 ended with the best prompt at 70.8 per cent exact and 464 prompt tokens, short of the 90 per cent target.
**Today's question:** can we teach the model our task, so it reads messy commands better and needs only a one-line prompt?
**What you will do:**
- Build 1,200 training examples and train a small adapter on the 1.5-billion-parameter model, kept in 4 bits.
- Measure the memory of three training settings, and watch the loss fall.
- Test on the same 72 messages and compare with Lab 16.

**What you will see:** in our test runs the fine-tuned model got 86 to 92 per cent exact with a one-line prompt of 46 tokens, against 464 tokens for the best prompt.
**Why it matters on a device:** a model that already knows the task needs a tiny prompt, which saves tokens, time and memory on every request.

**How to run this lab**
- Click **Kernel, Restart Kernel and Run All Cells**. Write your guess on paper before each cell marked **Predict**.
- The cells take about **1.5 minutes** in all (the training cell is about 1 minute of that), after the model has downloaded.
- If a cell takes longer than that, wait until the time is over, then interrupt the kernel and ask the trainer.
- Before you start: **Lab 16 must have been run**. The first cell prints its row of the scoreboard ("1 prompt only"). If the scoreboard is empty, run Lab 16 first.
- Training accuracy changes from run to run (we saw 86 to 92 per cent), because the examples are mixed in a random order. That is normal.

**The idea, in three pictures** (we use the same pictures in the slides)
- **Fine-tuning** is like an assistant who already speaks English joining your company. They do not learn the language again: they learn *your* commands from examples. Lab 16 only described the task in a prompt; fine-tuning changes the model so that it knows the task. *In one sentence: we teach the model by showing it examples.*
- **LoRA** is "do not rewrite the textbook, add sticky notes in the margin". The big weights stay frozen. Next to each big matrix we add two thin matrices, **A** and **B**, and only these are trained. *In one sentence: we train a small add-on, not the whole model.*
- **QLoRA** is LoRA, and also keeping the big book in a compressed form (4 bits) while the sticky notes stay normal. *In one sentence: the frozen model is stored small, so training fits in a small memory.*

**Why memory matters.** Training a model needs much more memory than running it. That is why we use QLoRA.

**Python in this lab:** loops, lists, dictionaries and function calls. The training loop itself is in `edge_train.py` (open it only if you are curious). The settings you can change are in the notebook.

In [ ]:
import sys, os
for folder in ["..", "../.."]:                  # find the course folder (the one that holds slm_lab.py)
    if os.path.exists(os.path.join(folder, "slm_lab.py")):
        sys.path.insert(0, os.path.abspath(folder))
        break
import torch
import pandas as pd
import edge_lab as el
import edge_nlu as nlu
import edge_train as et
from slm_lab import check, OUT
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
et.check_gpu()                                   # stops here with a plain message if this GPU is too small
print("scoreboard so far:", el.load_scoreboard())

**You should see** the GPU name, the line `This GPU can run Lab 17.` and the first row of the scoreboard from Lab 16 (the best prompt, its accuracy and its tokens). If the scoreboard is empty, run Lab 16 first.

If the lab stops here with a message, your Pod's GPU is too small or too old for the training: show the message to the trainer. Do not try to work around it.

## Part 1: understand it (no waiting)
Memory budget, the training data, the 4-bit model and LoRA by hand. Nothing here takes long.

## 17.1 The memory budget
A small device has a small memory. We **pretend** to have one by telling PyTorch it may use only a part of the GPU. If a setting needs more than the budget, it fails: exactly what would happen on the device.

🔧 Change the budget (as a share of the GPU's memory).

In [ ]:
BUDGET_SHARE = 0.30                             # 🔧 30 percent of this GPU (never less than 5.5 GB, see below)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
budget_gb, share = et.budget_for(total_gb, BUDGET_SHARE)
torch.cuda.set_per_process_memory_fraction(share)
print("GPU memory:", round(total_gb, 1), "GB")
print("our budget:", budget_gb, "GB")

Training keeps three things in memory for every parameter (every number of the model):
- the **weight** itself (2 bytes in 16 bits);
- its **gradient**: how much, and in which direction, this weight should change after a mistake (2 bytes);
- two helper numbers kept by the **optimiser** (the part that applies the changes; ours is called Adam), 4 bytes each.

**Predict.** Write on paper how many bytes per parameter a full fine-tune needs in total (choose 2, 6, 12 or 20), and how many GB that makes for 1.5 billion parameters. Then run the cell.

In [ ]:
PARAMETERS = 1_540_000_000                      # the model has about 1.5 billion numbers
cases = [
    ("full fine-tune, 16-bit", 2 + 2 + 8),      # weight + gradient + Adam's two 32-bit numbers
    ("LoRA, 16-bit weights", 2),                # weights only (the trained part is tiny)
    ("QLoRA, 4-bit weights", 0.5),              # weights only, in 4 bits
]
for name, bytes_per_parameter in cases:
    print(name, "->", round(PARAMETERS * bytes_per_parameter / 1e9, 1), "GB (before activations)")

**You should see** about 18 GB for a full fine-tune (12 bytes per parameter), 3 GB for LoRA and under 1 GB for QLoRA. The full fine-tune cannot fit our budget of about 7.6 GB. QLoRA can.

*In one sentence: a full fine-tune needs about six times the memory of simply holding the model; QLoRA needs far less.*

> **Optional: if you are curious.** These are rules of thumb. They leave out the **activations**: the numbers each layer produces while training, which the computer must keep for the backward step. They grow with the batch size and the length of the examples. We measure them in 17.5.

In [ ]:
check(budget_gb > 0, "Checkpoint: the memory budget is set.",
      "Run the cells of 17.1 again, from the top of the notebook.")

## 17.2 The training data
We do not collect thousands of real commands. We **build** them from facts: an intent plus its details (slots), said in many different ways (English, Hinglish, and text with speech-recognition mistakes). The generator is in `edge_nlu.py`. **None of the 72 test messages is in this set.** Each example is a short chat: a message, and the correct answer.

In [ ]:
main_examples = nlu.make_training_examples(1200)
counts = {}
for example in main_examples:
    counts[example["intent"]] = counts.get(example["intent"], 0) + 1
print(len(main_examples), "examples:", counts)
for example in main_examples[:5]:
    print(example["text"], "->", nlu.answer_text(example["intent"], example["slots"]))

**You should see** about 1,200 examples spread over the nine intents, with a mix of clean, noisy and Hinglish text.

**Replay data.** When we teach a new skill, the model can **forget** old ones. To protect them we mix in a few ordinary questions with ordinary answers. We call this **replay data**. They use a different system message, so the model learns "JSON only for the assistant task, normal answers for the rest". *In one sentence: replay data reminds the model of what it already knew.*

We also keep a small part of the examples aside as **validation** examples. The model never trains on them, so they show whether it is learning or only memorising (more in 17.6).

In [ ]:
replay_examples = nlu.make_replay_examples()
train_examples, validation_examples = et.mix_and_split(main_examples, replay_examples)
print("replay examples:", len(replay_examples), "(each used 3 times)")
print("train:", len(train_examples), " validation:", len(validation_examples))
print(replay_examples[0]["messages"][1]["content"], "->", replay_examples[0]["messages"][2]["content"])

### What the model is trained on
Each example is a chat. The model must learn to **write the assistant's answer**, not to write the user's message. The **loss** is a number that says how wrong the model's answer is (zero would be perfect). We compute it **only on the answer tokens**: the tokens of the prompt get the label `-100`, which means "ignore".

In [ ]:
tok = AutoTokenizer.from_pretrained(el.MODEL_ID)
rows = et.tokenise(tok, train_examples[:1])
tokens = rows[0]["input_ids"]
labels = rows[0]["labels"]
ignored = 0
for label in labels:
    if label == -100:
        ignored = ignored + 1
print("tokens in the example:", len(tokens), " ignored by the loss (prompt):", ignored, " learned (answer):", len(tokens) - ignored)
print("learned text:", tok.decode(tokens[ignored:]))

**You should see** that most of the tokens are ignored and only the JSON answer (plus the end marker) is learned. If the prompt were also learned, the model would waste effort learning to *write user messages*.

In [ ]:
check(len(train_examples) > 0 and len(validation_examples) > 0, "Checkpoint: training and validation examples are ready.",
      "Run the cells of 17.2 again, in order.")

## 17.3 Load the model in 4 bits
**Quantisation** means rounding every number to fewer levels, like measuring in millimetres, centimetres or whole metres. Fewer levels need fewer **bits** (a bit is one yes-or-no digit; 4 bits give 16 levels). Here the frozen big model is stored in 4 bits. We keep the frozen "big book" in a compressed form, while the sticky notes (LoRA) stay normal.

*In one sentence: 4 bits make the frozen model about a third of its size, so training fits.* Lab 18 quantises for real, for deployment. Today it only makes **training** fit.

> **Optional: if you are curious.** The 4-bit format is called **NF4**: it spaces its 16 levels to suit the usual spread of model weights. The arithmetic is still done in 16 bits (`bfloat16`).

In [ ]:
four_bit = BitsAndBytesConfig(
    load_in_4bit=True,                          # keep the weights in 4 bits
    bnb_4bit_quant_type="nf4",                  # the 4-bit format made for model weights
    bnb_4bit_compute_dtype=torch.bfloat16,      # but do the arithmetic in 16 bits
)
torch.cuda.reset_peak_memory_stats()
model = AutoModelForCausalLM.from_pretrained(el.MODEL_ID, quantization_config=four_bit, device_map="auto")
print("weights in memory:", el.model_size_mb(model), "MB   (in Lab 16 the same model took about 3,000 MB)")
print("peak GPU memory so far:", el.memory_gb(), "GB")

**You should see** the weights at roughly a third of their Lab 16 size. In our test run: about 1,122 MB, against about 3,087 MB in Lab 16.

In [ ]:
check(el.model_size_mb(model) < 2000, "Checkpoint: the model is loaded in 4 bits (far below 3,000 MB).",
      "The model is still large. Run the 4-bit cell of 17.3 again, after a Kernel restart.")

## 17.4 Attach LoRA
Remember the picture: **do not rewrite the textbook, add sticky notes in the margin.** **LoRA** freezes every original weight matrix `W` and adds two thin matrices, **A** and **B**, next to it. Only A and B are trained. The small add-on we get is called the **adapter**.

The number **r** (the **rank**) says how thin A and B are: a small `r` means a small adapter. *In one sentence: a bigger rank can learn more, and costs more memory.*

> **Optional: if you are curious.** The layer computes `W·x + (alpha / r) · B·A·x`. The setting `alpha` only scales the size of the sticky notes' effect.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)   # gets a 4-bit model ready for training
lora = LoraConfig(
    r=16,                                       # 🔧 rank: how wide A and B are (try 4 or 64)
    lora_alpha=32,                              # scale: the update is multiplied by alpha / r
    lora_dropout=0.05,                          # a little random dropout against over-fitting
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
trainable, total = et.count_parameters(model)
print("trained numbers:", trainable, "of about", PARAMETERS, "=", round(100 * trainable / PARAMETERS, 1), "%")

**You should see** that only about 1 per cent of the numbers are trained (in our test run, about 18.5 million, 1.2 per cent). The rest stay frozen.

**See the idea with tiny numbers.** One layer, with a 4 × 4 matrix `W`, and `r = 1`. You do not need to follow the arithmetic; look at which cells of the matrix change.

In [ ]:
import numpy as np
W = np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]])
B = np.array([[0.5], [0.0], [0.0], [0.2]])      # 4 x 1
A = np.array([[1.0, 0.0, 2.0, 0.0]])            # 1 x 4
alpha, r = 2, 1
new_W = W + (alpha / r) * (B @ A)               # B @ A is a 4 x 4 matrix of the same shape as W
print(new_W)
print("numbers in W:", W.size, " numbers in A and B:", A.size + B.size)

**You should see** that `W` changes only in the places where `B·A` is not zero, and that A and B together hold fewer numbers than W. For the real layers of this model (about 1,536 by 1,536) and `r = 16`, A and B hold about 2 per cent of the numbers of `W`.

In [ ]:
check(trainable > 0 and trainable < PARAMETERS / 10, "Checkpoint: LoRA is attached and only a small part of the model will be trained.",
      "Run the LoRA cell of 17.4 once only. If you ran it twice, restart the kernel and run all cells again.")

## Part 2: train it and measure it
**You now have:** a budget, 1,200 training examples, a 4-bit model with LoRA attached, and an idea of how A and B work. Now we measure the memory of three settings, train, and test. This part uses the GPU and takes a little longer.

## 17.5 Memory: which setting fits?
Training memory depends on how you train. We measure the **peak GPU memory** of three very short runs (3 steps each). A **batch** is the group of examples the model looks at together before one update.

**Predict.** Write on paper: (a) which of the three settings uses the least memory, and (b) whether the first one (batch 16) will fit our budget of about 7.6 GB. Then run the cell.

In [ ]:
train_rows = et.tokenise(tok, train_examples)
model.enable_input_require_grads()              # needed for checkpointing with a frozen base model
memory_table = []
settings = [
    ("batch 16", 16, 1, False),
    ("batch 16 + checkpointing", 16, 1, True),
    ("batch 4, update every 4 batches", 4, 4, False),
]
for name, batch_size, accumulate, checkpointing in settings:
    run = et.train_run(model, tok, train_rows, batch_size=batch_size, grad_accum=accumulate, max_steps=3,
                       checkpointing=checkpointing, run_name="probe")
    memory_table.append({"setting": name, "peak GB": run["peak_gb"], "seconds for 3 steps": run["seconds"]})
pd.DataFrame(memory_table)

**You should see** that the first setting may **not fit** the budget ("out of memory": exactly what a small device would do), while the other two fit. In our test run: batch 16 ran out of memory, batch 16 with checkpointing peaked at 4.89 GB, and batch 4 with accumulation peaked at 3.07 GB (the least). The result depends on your GPU's size.

Two tricks, one sentence each:
- **Checkpointing** recomputes some numbers instead of storing them: less memory, a little more time.
- **Accumulation** processes a few small batches before one update: the same learning with a smaller memory peak.

> **Optional: if you are curious.** Longer examples would need more memory too: the length of the example is one more thing you control.

In [ ]:
check(len(memory_table) == 3, "Checkpoint: you measured the three settings.",
      "Run the memory cell of 17.5 again. If it stops with an error other than out of memory, tell the trainer.")

## 17.6 Train for real
We make two passes over all the training data. One pass is one **epoch**, so this is 2 epochs. The `loss` (how wrong the answers are) should fall. Write down the last loss value and the peak memory that the cell prints.

🔧 Learning rate and epochs are the two settings that matter most.

In [ ]:
BATCH_SIZE = 8
EPOCHS = 2                                      # 🔧 passes over the data
LEARNING_RATE = 2e-4                            # 🔧 how big each update is
result = et.train_run(model, tok, train_rows, batch_size=BATCH_SIZE, epochs=EPOCHS, learning_rate=LEARNING_RATE,
                      checkpointing=True, run_name="intent")       # checkpointing keeps us inside the budget
print("steps:", result["steps"], " seconds:", result["seconds"], " peak GPU memory:", result["peak_gb"], "GB")
print("loss:", result["losses"])

**You should see** the loss falling quickly to a very small number, with a peak memory **inside the budget**. In our test run: 290 steps, about 60 seconds, peak 3.69 GB on an RTX 4090 (the time depends on your GPU). The whole training ran in a memory that a small device could offer.

**Is a tiny loss good news?** Not by itself. The model can start to **memorise** the training examples instead of learning the task. We kept the validation examples aside. Look at the loss on them in the next cell, and then remember where they came from.

In [ ]:
validation_rows = et.tokenise(tok, validation_examples)
total_loss = 0
model.eval()
for row in validation_rows[:40]:
    batch = et.collate([row])
    with torch.no_grad():
        output = model(input_ids=batch["input_ids"].cuda(), attention_mask=batch["attention_mask"].cuda(), labels=batch["labels"].cuda())
    total_loss = total_loss + float(output.loss)
print("loss on 40 validation examples:", round(total_loss / 40, 4))

**You should see** a validation loss that is also tiny (in our test run, 0.0002). But these examples were made by the **same generator** as the training examples, so they are easy. A small loss only says the model learned *this* kind of text. *In one sentence: a small loss is a good sign, not a proof.* **The real test is messages it has never seen: the 72 golden messages.**

In [ ]:
check(len(result["losses"]) > 0 and result["steps"] > 0, "Checkpoint: the training ran and recorded its loss.",
      "Run the training cell of 17.6 again. If it says out of memory, lower BATCH_SIZE to 4.")

## 17.7 Save the adapter
The adapter is small: only A and B (the sticky notes). You can keep it, share it, and attach it to the same base model any time. In our test run it was about 74 MB, against about 3,000 MB for the model.

In [ ]:
ADAPTER_FOLDER = str(OUT / "edge_adapter")
model.save_pretrained(ADAPTER_FOLDER)
size_mb = 0
for name in os.listdir(ADAPTER_FOLDER):
    size_mb = size_mb + os.path.getsize(os.path.join(ADAPTER_FOLDER, name)) / 1e6
print("adapter folder:", ADAPTER_FOLDER)
print("adapter size:", round(size_mb), "MB   (the base model is about 3,000 MB)")

In [ ]:
check(os.path.exists(ADAPTER_FOLDER), "Checkpoint: the adapter folder exists.",
      "Run the cell above again. Lab 18 needs this folder (or it uses the prepared adapter).")

## 17.8 After: the same 72 messages
We free the training model, load the 16-bit model with the adapter attached, and test it with a **one-line prompt**: the prompt it was trained with. **Predict.** Write your guess for the exact % (above or below the 70.8 of Lab 16's best prompt?) and for the prompt tokens (about 40, 100 or 400?).

In [ ]:
del model
torch.cuda.empty_cache()
tok, tuned = el.load_edge_model(adapter=ADAPTER_FOLDER)
after = el.evaluate(tok, tuned, nlu.SHORT_SYSTEM)
print("fine-tuned, one-line prompt:", after["exact %"], "% exact,", after["prompt tokens"], "prompt tokens,", after["ms per message"], "ms per message")

In [ ]:
before = el.load_scoreboard()["1 prompt only"]
rows = [
    {"model": "Lab 16: best prompt", "exact %": before["exact_percent"], "prompt tokens": before["prompt_tokens"], "ms per message": before["ms_per_message"]},
    {"model": "fine-tuned, one line", "exact %": after["exact %"], "prompt tokens": after["prompt tokens"], "ms per message": after["ms per message"]},
]
pd.DataFrame(rows)

**You should see** the fine-tuned model **more accurate and with far fewer prompt tokens** than the best prompt. In our test runs: 86 to 92 per cent exact, and 46 prompt tokens against 464, about a tenth of the length.

*If it looks different:* accuracy varies from run to run (our four runs gave 86.1, 88.9, 88.9 and 91.7 per cent), so do not worry if yours is a few points away. The time per message may not be lower than in Lab 16, because the adapter is still a separate step: **Lab 18 merges it**.

Which kinds of message improved most, and is the brief met? Look at the next cell.

In [ ]:
for kind in after["by_kind"]:
    print(kind, ":", after["by_kind"][kind], "%")
for mistake in after["wrong"][:5]:
    print(nlu.describe_mistake(mistake))

**You should see** the accuracy for each kind of message, and a short list of what is still wrong. Compare with your own table of 16.6. In one of our runs: clean 87.5, noisy 90.0, Hinglish 85.7, short 100, manual 87.5 and out_of_domain 62.5 per cent, so the messages outside the assistant's job (such as TV commands) were the weakest.

In our test runs the overall accuracy was between 86 and 92 per cent: well above the best prompt, and **right around the 90 per cent of the brief**. Those remaining mistakes are the work of **Lab 20** (what to do when the intent is wrong).

In [ ]:
check(after["exact %"] > before["exact_percent"], "Checkpoint: the fine-tuned model beats the best prompt of Lab 16.",
      "Your accuracy is not above Lab 16. Look at the wrong messages above; if the training cell showed an error, run 17.6 again.")

## 17.9 What did the model forget?
A fine-tuned model can lose old skills. We ask five ordinary questions **without** the assistant's system message and see whether it still answers in plain language.

In [ ]:
questions = ["What is the capital of Japan?", "Write one sentence about the sea.", "What is five times six?",
             "Name a fruit that is red.", "Say hello in one short sentence."]
prompt_texts = []
for question in questions:
    messages = [{"role": "system", "content": "You are a helpful assistant. Answer briefly."}, {"role": "user", "content": question}]
    prompt_texts.append(tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
replies = el.ask_many(tok, tuned, prompt_texts, max_new_tokens=40)["replies"]
for index in range(len(questions)):
    print(questions[index], "->", replies[index])

**You should see** normal sentences, not JSON, so the replay examples did their job. (We did not train a version without replay data in this lab; models trained that way often start answering everything in JSON. Try it only if you have time.)

In [ ]:
check(len(replies) == 5, "Checkpoint: you asked the five ordinary questions.",
      "Run the cell above again.")

## 17.10 Your turn: how big is the adapter?
Write the formula for how many numbers one LoRA layer holds. `A` has `r` rows and `d_in` columns. `B` has `d_out` rows and `r` columns. Then use it to see how the rank changes the size. Write down the three numbers the cell prints (one for each `r`). If you are stuck, open the solution cell below the starter.

In [ ]:
def lora_numbers(d_in, d_out, r):
    return 0                                    # TODO: replace 0 with the number of numbers in A and B together

for r in [4, 16, 64]:
    print("r =", r, "->", lora_numbers(1536, 1536, r), "numbers in one layer")

In [ ]:
check(lora_numbers(1536, 1536, 16) == 49152, "Right: 16 x 1,536 for A plus 1,536 x 16 for B is 49,152 numbers. Doubling r doubles the adapter.",
      "A has r x d_in numbers and B has d_out x r numbers. Add them.")

## 17.11 What you proved: the second row

In [ ]:
el.save_row("2 fine-tuned", {"what": "1.5B + LoRA adapter, one-line prompt", "exact_percent": after["exact %"],
                            "prompt_tokens": after["prompt tokens"], "answer_tokens": after["answer tokens"],
                            "ms_per_message": after["ms per message"], "weights_mb": el.model_size_mb(tuned)})
board = el.load_scoreboard()
for step in board:
    print(step, board[step])
check(len(board) >= 2, "Checkpoint: the scoreboard has two rows (prompt only, fine-tuned).",
      "Run the cell of 17.11 again. If a row is missing, run Lab 16 first.")

**You should see** two rows. We started with the cheapest change (a prompt) and then changed the model. The model still takes about 3,000 MB, though. **Lab 18** makes it small.

## 17.12 Your turn (about 10 minutes): teach it your own phrases
In Lab 16 you wrote messages of your own. Now we see how the **fine-tuned** model does on them, then **teach it more of your wording** and test again. This is how a real product improves: look at what fails, add examples of that kind, train again, measure again.

**Step 1: how does the fine-tuned model do on your messages?**

In [ ]:
my_cases = nlu.load_my_messages()
if len(my_cases) == 0:
    print("No messages found. Run section 16.9 of Lab 16 first: it saves your messages.")
before_mine = el.evaluate_cases(tok, tuned, nlu.SHORT_SYSTEM, my_cases)
nlu.show_results(my_cases, before_mine["replies"])
print("fine-tuned model, your messages: exact", before_mine["exact %"], "%")

**Step 2: write new training examples.** Look at the `XX` lines above (if there are none, think of your hardest message). Write **4 new messages in the same style**: the same kind of wording, but different words, with their right answers. They must **not** be your test messages: a test must use messages the model never saw (the lab leaves out any that are the same). Each example you write is shown to the model 8 times in training. Write them in the same style as the messages that failed: if a Hinglish air-conditioner command failed, write other Hinglish air-conditioner commands, not a light command.

In [ ]:
MY_EXAMPLES = [
    {"text": "put on the study light", "intent": "set_light", "slots": {"room": "study", "state": "on"}},   # an example
    {"text": "", "intent": "", "slots": {}},      # TODO 1: a new message like the ones that failed, and its right answer
    {"text": "", "intent": "", "slots": {}},      # TODO 2
    {"text": "", "intent": "", "slots": {}},      # TODO 3
    {"text": "", "intent": "", "slots": {}},      # TODO 4
]
my_examples = nlu.prepare_my_messages(MY_EXAMPLES, kind="train")

**Step 3: train again (about a minute).** The next cell frees the GPU, carries on training the adapter you made above, on your examples plus a small slice of the standard data (so it learns your wording and does not forget), and saves it as `edge_adapter_mine`.

In [ ]:
del tuned
torch.cuda.empty_cache()
trained = et.train_adapter(my_examples, "edge_adapter_mine", test_cases=my_cases, start_from=ADAPTER_FOLDER)
print("your examples used:", trained["used"], "| training took", trained["seconds"], "seconds | peak GPU memory", trained["peak_gb"], "GB")

**Step 4: test again, on your messages and on the 72.** Both numbers matter: your messages tell you whether the model **learned** your wording, and the 72 tell you whether it **forgot** anything.

In [ ]:
tok, mine = el.load_edge_model(adapter=trained["folder"])
after_mine = el.evaluate_cases(tok, mine, nlu.SHORT_SYSTEM, my_cases)
nlu.show_results(my_cases, after_mine["replies"])
golden_mine = el.evaluate(tok, mine, nlu.SHORT_SYSTEM)
print("your messages  :", before_mine["exact %"], "% ->", after_mine["exact %"], "%")
print("the 72 messages:", after["exact %"], "% ->", golden_mine["exact %"], "%")

In [ ]:
check(os.path.exists(trained["folder"]), "Checkpoint: your own adapter is saved and tested.",
      "The adapter folder is missing. Run the training cell of Step 3 again.")

**You should see** two lines of before and after. Your messages usually improve when your new examples have the same pattern as the ones that failed. The 72 should stay about where they were: if they drop a lot, the model forgot something. Numbers can also stay the same or move a little by chance: training varies from run to run, and with only a few messages one message is a big step. **Write one sentence: what did your examples change, and how do you know?** Remember that with six messages **one message is worth 16 points**, so look at *which* messages changed, not only at the percentage. If the model answers with an intent that is not one of our nine, the lab shows it as `unreadable`: the model invented a label. That is a real failure, and it is why Lab 20 checks the answer before acting.

## What you take away
- Fine-tuning teaches the model our task from examples. QLoRA keeps the big model frozen in 4 bits and trains only a small adapter (the sticky notes), so it fits in a small memory.
- In our test runs the fine-tuned model reached 86 to 92 per cent with a one-line prompt of 46 tokens, against 70.8 per cent and 464 tokens for the best prompt.
- A small loss is not proof. Always measure before and after on a test set the model never saw.

**Good habits from this lab:**
1. Compute the loss on the **answer** only.
2. Mix in **replay** data so the model does not forget its old skills.
3. Keep **validation** examples aside and watch both losses.

## Check your understanding
1. Why does QLoRA fit in a small memory when a full fine-tune does not?
*✍️ Your answer:* 
2. Why do the prompt tokens get the label -100?
*✍️ Your answer:* 
3. What is replay data for, and what did you see without it?
*✍️ Your answer:* 
4. The adapter has r = 16. What happens to its size if you use r = 64, and why?
*✍️ Your answer:* 

## Next
**Lab 18:** the model is accurate but still about 3,000 MB. We make it small, for real, and run it with no GPU at all.